# Phase 2 - Part 2: Siting refined: orientation sweep + fine downscaling

1. Synthetic resource year on the coarse sea grid.
2. Coarse optimise with a 4-orientation sweep.
3. Downscale the chosen region to 1.3 km and optimise precisely.
4. Submission.

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
import json, time
import numpy as np, pandas as pd
import synthetic_generator as sg, zone, downscaling as dsc, target_loader as al
import optimization as opt
from wind_farm_simulator import grid_layout, WindSeries, validate_layout
from turbines_catalog import get_spec
from synth_wind import SynthWindCache
HUB_M = 170.0; N_TURB = 55; BOX_M = 15000.0; TURBINE0 = 'IEA_22MW'
from shear import SHEAR_ALPHA

## 1. Synthetic resource year (AR) on the coarse sea grid

In [ ]:
hist = sg.load_coarse_history()
synth = sg.sample_ar_year(sg.fit_ar_generator(hist), seed=0)
coarse_cache = SynthWindCache(synth, hub_height_m=HUB_M)
print('synthetic year:', synth.U.shape[0], 'steps,', synth.U.shape[1], 'sea cells')

## 2. Coarse optimise - brute scan with the 4-orientation sweep

In [ ]:
spec0 = get_spec(TURBINE0)
lx0, ly0 = grid_layout(N_TURB, spacing_d=7, diameter_m=spec0.diameter_m, rotation_deg=0)
(lat_b, lon_b) = zone.zone_bounds()
lats = np.linspace(lat_b[0], lat_b[1], 8); lons = np.linspace(lon_b[0], lon_b[1], 8)
cands = [(la, lo) for la in lats for lo in lons]
t0 = time.time()
coarse = opt.scan_placement_grid(
    candidates=cands, layout_x_m=lx0, layout_y_m=ly0, turbine_key=TURBINE0,
    wind_cache=coarse_cache, is_allowed=lambda la, lo: zone.is_in_allowed_zone(la, lo, max_depth_m=50))
clat, clon = coarse.best_config.centre_lat, coarse.best_config.centre_lon
print(f'scanned {coarse.n_evaluations} allowed candidates (x4 orient) in {time.time()-t0:.0f}s')
print(f'best coarse: ({clat:.2f},{clon:.2f}) AEP={coarse.best_aep_gwh:.0f} GWh '
      f'orient={coarse.best_orientation_deg:.0f}°')
print(pd.DataFrame(coarse.log).head(5)[['lat','lon','aep_gwh','orientation_deg']].to_string(index=False))

## 3. Downscale to 1.3 km, then place turbines individually (QD)

In [ ]:
dwn = dsc.train_downscaler(
    [d.date() for d in pd.date_range('2020-02-01', '2020-02-12', freq='2D')], hours=(12,))
lat1d, lon1d = dsc._reanalysis_axes()
_li = {round(float(x), 3): k for k, x in enumerate(lat1d)}
_lj = {round(float(x), 3): k for k, x in enumerate(lon1d)}

def _scatter(U_t, V_t):
    u = np.full((lat1d.size, lon1d.size), np.nan); v = u.copy()
    for la, lo, uu, vv in zip(synth.lat, synth.lon, U_t, V_t):
        i = _li.get(round(float(la), 3)); j = _lj.get(round(float(lo), 3))
        if i is not None and j is not None: u[i, j] = uu; v[i, j] = vv
    return u, v

st = al.load_static(); alat = np.asarray(st.lat); alon = np.asarray(st.lon)
win = (np.abs(alat - clat) < 0.35) & (np.abs(alon - clon) < 0.45) & st.sea
idx = np.where(win.ravel())[0]
n_sub = 180; stride = max(1, synth.U.shape[0] // n_sub)
sel = list(range(0, synth.U.shape[0], stride))
t0 = time.time(); FU, FV = [], []
for t in sel:
    cu, cv = _scatter(synth.U[t], synth.V[t])
    fu, fv = dsc.downscale(dwn, cu, cv)
    FU.append(fu.ravel()[idx]); FV.append(fv.ravel()[idx])
FU = np.asarray(FU); FV = np.asarray(FV)
print(f'downscaled {len(sel)} steps over {idx.size} fine cells in {time.time()-t0:.0f}s')

In [ ]:
class FineSynthCache:
    """Serve hub-height WindSeries at any (lat, lon) from the downscaled fine window."""
    def __init__(self, lat, lon, U, V, times, hub_m=HUB_M, ref_m=125.0, alpha=SHEAR_ALPHA):
        self.lat, self.lon = np.asarray(lat), np.asarray(lon)
        self.U, self.V, self.times = U, V, times
        self.shear = (hub_m / ref_m) ** alpha
        self._c = {}
    def get(self, lat, lon):
        key = (round(float(lat), 3), round(float(lon), 3))
        if key in self._c: return self._c[key]
        k = int(np.argmin((self.lat - lat) ** 2 + (self.lon - lon) ** 2))
        u, v = self.U[:, k] * self.shear, self.V[:, k] * self.shear
        ws = WindSeries(pd.DataFrame({'time': self.times,
            'ws': np.hypot(u, v).astype('float32'),
            'wd': ((270.0 - np.degrees(np.arctan2(v, u))) % 360.0).astype('float32')}))
        self._c[key] = ws; return ws

fine_lat = alat.ravel()[idx]; fine_lon = alon.ravel()[idx]
fine_cache = FineSynthCache(fine_lat, fine_lon, FU, FV, synth.times[sel])
fb_lat = (float(fine_lat.min()), float(fine_lat.max()))
fb_lon = (float(fine_lon.min()), float(fine_lon.max()))

place = opt.optimize_placement(
    layout_x_m=lx0, layout_y_m=ly0, turbine_key=TURBINE0, wind_cache=fine_cache,
    lat_bounds=fb_lat, lon_bounds=fb_lon, orientations=opt.ORIENTATIONS, max_iter=12, seed=0)
flat, flon = place.best_config.centre_lat, place.best_config.centre_lon

tkey, n_t = TURBINE0, N_TURB   # fixed: 55 x IEA_22MW

qd = opt.optimize_layout_qd(
    centre_lat=flat, centre_lon=flon, turbine_key=tkey, wind_cache=fine_cache,
    n_turbines=n_t, box_size_m=BOX_M, min_spacing_d=5.0, n_iters=2500, seed=0)
best = qd.best.best_config
cap = n_t * get_spec(tkey).rated_power_mw
print(f'fine centre ({flat:.3f},{flon:.3f}) orient={place.best_orientation_deg:.0f}\u00b0')
print(f'FIXED {tkey} x{n_t} = {cap:.0f} MW')
print(f'QD individual placement: AEP={qd.best.best_aep_gwh:.0f} GWh '
      f'CF={qd.best.best_capacity_factor:.3f} | archive {len(qd.cells)} layouts, '
      f'{qd.n_evaluations} sims in {qd.elapsed_seconds:.0f}s')

## 4. Submission - validate & export

In [ ]:
spec_b = get_spec(best.turbine_key)
ok, errs = validate_layout(best.layout_x_m, best.layout_y_m, box_size_m=BOX_M,
                           max_turbines=55, min_spacing_d=5.0, diameter_m=spec_b.diameter_m)
print('layout valid:', ok, errs)
opt.export_submission(best, Path('part2_siting/submission_orientation_sweep.json'), team='reference')
print('wrote submission_orientation_sweep.json')
print(json.dumps(json.loads(Path('part2_siting/submission_orientation_sweep.json').read_text()), indent=2)[:400])